In [1]:
from pathlib import Path 
import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['MKL_SERVICE_FORCE_INTEL'] = '1'
from tqdm import tqdm
from IPython.display import Video

import torch
import numpy as np

import sys
sys.path.append("/srv/sferraro/choreographer/")

import envs

import matplotlib.pyplot as plt
import matplotlib.animation as animation

/home/elephant/.local/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/elephant/miniconda3/envs/urlb/lib/python3.8/site-packages/robosuite-1.4.0-py3.8.egg/robosuite/__init__.py:7: DeprecationWarning: The 'warn' method is deprecated, use 'warning' instead
  ROBOSUITE_DEFAULT_LOGGER.warn("No private macro file found!")
[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /home/elephant/miniconda3/envs/urlb/lib/python3.8/site-packages/robosuite-1.4.0-py3.8.egg/robosuite/scripts/setup_macros.py (__init__.py:9)


In [2]:
# Import agent model (WM + Actor Critic)
agent_path = Path(f'/srv/sferraro/choreographer/exp_local/2023.03.29/121720_dreamer/last_snapshot.pt')

def load_agent(agent_path):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    with agent_path.open('rb') as f:
        obj = torch.load(f, map_location=torch.device(device))
        agent = obj['agent']
        step = obj['_global_step']
        agent.device = device
        agent.wm.device = device
        agent.wm.rssm.device = device
        agent.wm.rssm._cell.device = device
    return agent, step

agent, global_step = load_agent(agent_path)

In [3]:
from hydra import compose, initialize
from omegaconf import OmegaConf

initialize(config_path="../exp_local/2023.03.29/141300_dreamer/.hydra/", job_name="config")
cfg = compose(config_name="config")

In [4]:
# Agent parametrization
obs_type = agent.cfg.obs_type
action_repeat = agent.cfg.action_repeat
snapshot_ts = global_step * action_repeat

agent.reward_free = True

agent.use_selector = False
agent.detached_exploration = True

seed = agent.cfg.seed

task = "panda_Stack"
domain = task.split("_")[0]

# Env creation
eval_env = envs.make(task, obs_type, frame_stack=1, 
                    action_repeat=action_repeat, seed=seed, env_config=cfg.env)

/home/elephant/miniconda3/envs/urlb/lib/python3.8/site-packages/gym/spaces/box.py:84: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


In [6]:
# get first observation from env (reset), feed it to encoder and get feat, feed features to actor for planning sequence, swap features to test out different objects
import utils

render_size = 64
camera = "agentview"

eval_mode = True

agent_state = None
meta = agent.init_meta()

dreamer_obs = eval_env.reset()
episodes = 10

step, episode, total_reward = 0, 0, 0
for ep in tqdm(range(episodes)):
    agent_state = None
    dreamer_obs = eval_env.reset()
    
    with torch.no_grad(), utils.eval_mode(agent):
        while not bool(dreamer_obs['is_last']):
            action, agent_state = agent.act(
                                    dreamer_obs,
                                    meta,
                                    step,
                                    eval_mode=False,
                                    state=agent_state,
                                )
        
            dreamer_obs = eval_env.step(action)
            step += 1
    
    episode += 1


#     skill_z = torch.from_numpy(skill).to(agent.device).unsqueeze(0).unsqueeze(0)
#     skill_z = skill_z @ agent.skill_module.emb.weight.T

#     x = deter = agent.skill_module.skill_decoder(skill_z).mean

#     stats = agent.wm.rssm._suff_stats_ensemble(x)
#     index = torch.randint(0, agent.wm.rssm._ensemble, ()) 
#     stats = {k: v[index] for k, v in stats.items()}
#     dist = agent.wm.rssm.get_dist(stats)
#     stoch = dist.sample() 
#     prior = {'stoch': stoch, 'deter': deter, **stats}

#     decoder = agent.wm.heads['decoder']
#     openl = decoder(agent.wm.rssm.get_feat(prior))['observation'].mean.squeeze() 
#     skill_img = torch.clip(openl + 0.5, 0, 1).cpu().numpy()
#     skill_obs[n] = [1, skill_img.transpose(1,2,0), 0]


#     frame = eval_env.sim.render(
#         render_size, render_size, mode="offscreen", camera_name=train_env._camera
#     ).copy()

#     imagelist[n].append(frame)
#     rewardlist[n].append(time_step['reward'])
#     for _ in range(steps):
#         action, agent_state = agent.act(time_step, 
#                             meta,
#                             0,
#                             eval_mode=eval_mode,
#                             state=agent_state)
        

#         time_step = train_env.step(action)

#         if task == MW:
#             frame = train_env.sim.render(
#                 render_size, render_size, mode="offscreen", camera_name=train_env._camera
#             ).copy()
#         else:
#             frame = train_env.physics.render(height=render_size,
#                                         width=render_size,
#                                         camera_id=camera_id)
#         imagelist[n].append(frame)
#         rewardlist[n].append(time_step['reward'])
    
#         if time_step['is_last']:
#             time_step = train_env.reset()
#             agent_state = None

# for skill_id in range(skill_dim):
#     skill_obs[skill_id][2] /= steps*skill_dim

# if obs_type == 'pixels':
#     fig, axes = plt.subplots(rows, columns, figsize=(columns // rows * c_size,c_size)) # make figure

#     for index, (p_image, image, avg_image) in enumerate(skill_obs):
#         r = index // columns
#         c = index % columns
#         a = axes[r][c]
#         a.axis('off')
#         a.set_aspect('equal')
#         # t = a.set_title(f"Prob: {p_image:.3f}\n Avg: {avg_image:.3f}")
#         im = a.imshow(image, cmap=plt.get_cmap('jet'), vmin=0, vmax=255)
#     fig.tight_layout()

100%|██████████| 10/10 [00:29<00:00,  2.90s/it]


['custom_lift']


ModuleNotFoundError: No module named '__main__.custom_lift'; '__main__' is not a package